# AED-XAI Full Pipeline Runner
Interactive end-to-end execution, selector training, baseline comparison, and result export.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import gc
import json
import logging
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from tqdm.auto import tqdm

from src.detector import DetectorWrapper
from src.evaluator import AutoEvaluator, EvalResult
from src.pipeline import AEDXAIPipeline, PipelineResult
from src.utils import load_image, set_seed, setup_logging
from src.xai_methods import get_explainer

setup_logging("INFO")
set_seed(42)
logger = logging.getLogger("run_full_pipeline")

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA build: {torch.version.cuda}")

## Configuration Check

In [ ]:
CONFIG_PATHS = {
    "detector": PROJECT_ROOT / "config/detector_config.yaml",
    "vlm": PROJECT_ROOT / "config/vlm_config.yaml",
    "xai": PROJECT_ROOT / "config/xai_config.yaml",
    "eval": PROJECT_ROOT / "config/eval_config.yaml",
}

configs = {}
for name, path in CONFIG_PATHS.items():
    with path.open("r", encoding="utf-8") as handle:
        configs[name] = yaml.safe_load(handle) or {}

detector_cfg = configs["detector"]["detector"]
vlm_cfg = configs["vlm"]["vlm"]
xai_cfg = configs["xai"]["xai"]
eval_cfg = configs["eval"]["evaluation"]

enabled_methods = [name for name, cfg in xai_cfg["methods"].items() if cfg.get("enabled", False)]
print(f"Detector primary      : {detector_cfg['primary']['name']}")
print(f"Detector conf/nms     : {detector_cfg['conf_thresh']} / {detector_cfg['nms_thresh']}")
print(f"VLM model             : {vlm_cfg['model_name']}")
print(f"VLM quantization      : {vlm_cfg['quantization']}")
print(f"Enabled XAI methods   : {enabled_methods}")
print(f"Feedback max iters    : {eval_cfg['feedback']['max_iterations']}")
print(f"Composite weights     : {eval_cfg['composite_weights']}")

## Initialize Pipeline

In [ ]:
pipeline = AEDXAIPipeline(
    detector_config_path=str(CONFIG_PATHS["detector"]),
    vlm_config_path=str(CONFIG_PATHS["vlm"]),
    xai_config_path=str(CONFIG_PATHS["xai"]),
    eval_config_path=str(CONFIG_PATHS["eval"]),
)
pipeline.setup()
if torch.cuda.is_available():
    print(f"VRAM after setup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("Running without CUDA.")

## Sanity Test (1 Image)

In [ ]:
def pipeline_result_to_row(result: PipelineResult) -> dict:
    eval_results = list(result.evaluation_results)
    def mean_attr(attr: str) -> float:
        values = [float(getattr(item, attr)) for item in eval_results]
        return float(np.mean(values)) if values else 0.0
    return {
        "image_path": Path(result.image_path).name,
        "num_detections": len(result.detections),
        "composite_score": float(result.composite_score) if result.composite_score is not None else 0.0,
        "iterations": int(result.metadata.get("iterations", 0)),
        "converged": bool(result.metadata.get("converged", False)),
        "mean_pg": mean_attr("pg"),
        "mean_oa": mean_attr("oa"),
        "mean_ebpg": mean_attr("ebpg"),
        "mean_sparsity": mean_attr("sparsity"),
        "mean_insertion_auc": mean_attr("insertion_auc"),
        "mean_deletion_auc": mean_attr("deletion_auc"),
        "total_time": float(sum(float(item.computation_time) for item in eval_results)),
        "has_error": "error" in result.metadata,
    }

image_paths_all = sorted((PROJECT_ROOT / "data/coco/val2017").glob("*.jpg"))
assert image_paths_all, "No COCO val2017 images found. Run data download first."
image_path = image_paths_all[0]

result = pipeline.run_on_image(str(image_path))
print(f"Image: {image_path.name}")
print(f"Num detections: {len(result.detections)}")
print(f"Composite score: {result.composite_score}")
print(f"Iterations: {result.metadata.get('iterations', 0)}")
print(f"Converged: {result.metadata.get('converged', False)}")

image = load_image(str(image_path))
boxed = image.copy()
for det in result.detections:
    x1, y1, x2, y2 = det.bbox
    cv2.rectangle(boxed, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(boxed, f"{det.class_name} {det.confidence:.2f}", (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

if result.saliency_maps:
    saliency = result.saliency_maps[0].map
else:
    saliency = np.zeros(image.shape[:2], dtype=np.float32)
heatmap_uint8 = np.uint8(np.clip(saliency, 0, 1) * 255)
heatmap_rgb = cv2.cvtColor(cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_HOT), cv2.COLOR_BGR2RGB)
overlay = np.uint8(0.6 * image + 0.4 * heatmap_rgb)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(boxed)
axes[0].set_title("Detections")
axes[1].imshow(saliency, cmap="hot")
axes[1].set_title("Best Saliency")
axes[2].imshow(overlay)
axes[2].set_title("Overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

metrics_rows = [item.as_dict() for item in result.evaluation_results]
display(pd.DataFrame(metrics_rows).round(3))

## Generate Selector Training Data (Small, 50 Images)

In [ ]:
from scripts.train_selector import SelectorTrainingConfig, SelectorTrainingRunner

selector_config = SelectorTrainingConfig(
    detector_config=str(CONFIG_PATHS["detector"]),
    xai_config=str(CONFIG_PATHS["xai"]),
    eval_config=str(CONFIG_PATHS["eval"]),
    annotations_path=str(PROJECT_ROOT / "data/coco/annotations/instances_val2017.json"),
    images_dir=str(PROJECT_ROOT / "data/coco/val2017"),
    output_csv=str(PROJECT_ROOT / "results/notebook_selector_training_data.csv"),
    progress_csv=str(PROJECT_ROOT / "results/notebook_selector_training_progress.csv"),
    output_model_path=str(PROJECT_ROOT / "data/checkpoints/xai_selector.pth"),
    max_images=50,
    target_detections=250,
    checkpoint_every=25,
    oracle_mode=True,
    sample_seed=42,
)
runner = SelectorTrainingRunner(selector_config)
selector_results = runner.run()
print(selector_results)

## Run AED-XAI on 50 Images

In [ ]:
image_paths_50 = sorted((PROJECT_ROOT / "data/coco/val2017").glob("*.jpg"))[:50]
aedxai_results = []
for path in tqdm(image_paths_50, desc="AED-XAI 50 images"):
    try:
        aedxai_results.append(pipeline.run_on_image(str(path)))
    except Exception as exc:
        logger.warning("AED-XAI failed for %s: %s", path, exc, exc_info=True)
        aedxai_results.append(PipelineResult(image_path=str(path), metadata={"error": str(exc)}))

aedxai_df = pd.DataFrame([pipeline_result_to_row(item) for item in aedxai_results])
display(aedxai_df.describe(include="all"))

## Run Baseline on 50 Images (No Feedback, Fixed GradCAM)

In [ ]:
def eval_results_to_row(image_path: Path, num_detections: int, eval_results: list[EvalResult], has_error: bool = False) -> dict:
    def mean_attr(attr: str) -> float:
        values = [float(getattr(item, attr)) for item in eval_results]
        return float(np.mean(values)) if values else 0.0
    return {
        "image_path": image_path.name,
        "num_detections": num_detections,
        "composite_score": mean_attr("composite"),
        "iterations": 1,
        "converged": False,
        "mean_pg": mean_attr("pg"),
        "mean_oa": mean_attr("oa"),
        "mean_ebpg": mean_attr("ebpg"),
        "mean_sparsity": mean_attr("sparsity"),
        "mean_insertion_auc": mean_attr("insertion_auc"),
        "mean_deletion_auc": mean_attr("deletion_auc"),
        "total_time": float(sum(float(item.computation_time) for item in eval_results)),
        "has_error": has_error,
    }

baseline_detector = DetectorWrapper("yolox-s", config_path=str(CONFIG_PATHS["detector"]))
baseline_detector.load_model()
with CONFIG_PATHS["xai"].open("r", encoding="utf-8") as handle:
    baseline_xai_cfg = yaml.safe_load(handle)
baseline_explainer = get_explainer("gradcam", baseline_xai_cfg["xai"]["methods"]["gradcam"])
baseline_evaluator = AutoEvaluator(str(CONFIG_PATHS["eval"]))
baseline_target_layer = baseline_detector.get_target_layer()
baseline_rows = []

try:
    for path in tqdm(image_paths_50, desc="Baseline GradCAM"):
        try:
            image = load_image(str(path))
            detections = baseline_detector.detect(image)
            model = baseline_detector.get_model()
            eval_results = []
            for detection in detections:
                saliency = baseline_explainer.explain(model, image, detection, baseline_target_layer)
                eval_results.append(baseline_evaluator.evaluate_all(saliency, detection.bbox, model, image, detection))
            baseline_rows.append(eval_results_to_row(path, len(detections), eval_results))
        except Exception as exc:
            logger.warning("Baseline failed for %s: %s", path, exc, exc_info=True)
            baseline_rows.append(eval_results_to_row(path, 0, [], has_error=True))
finally:
    baseline_detector.unload_model()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df.describe(include="all"))

## Comparison Table

In [ ]:
def mean_std_text(series: pd.Series) -> str:
    return f"{series.mean():.3f} ± {series.std(ddof=0):.3f}"

comparison_rows = []
metric_map = {
    "composite": "composite_score",
    "pg": "mean_pg",
    "oa": "mean_oa",
    "sparsity": "mean_sparsity",
    "iterations": "iterations",
    "time (s/img)": "total_time",
}
for label, column in metric_map.items():
    comparison_rows.append({
        "Metric": label,
        "AED-XAI (mean±std)": mean_std_text(aedxai_df[column]),
        "Baseline GradCAM (mean±std)": "N/A (always 1)" if label == "iterations" else mean_std_text(baseline_df[column]),
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.style.set_properties(**{"text-align": "center"}).set_table_styles([{"selector": "th", "props": [("text-align", "center")]}]))

## Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

means = [aedxai_df["composite_score"].mean(), baseline_df["composite_score"].mean()]
stds = [aedxai_df["composite_score"].std(ddof=0), baseline_df["composite_score"].std(ddof=0)]
axes[0].bar(["AED-XAI", "GradCAM"], means, yerr=stds, capsize=5, color=["#C4AD66", "#4878CF"])
axes[0].set_ylabel("Composite Score")
axes[0].set_title("Composite Comparison")
axes[0].set_ylim(0, 1)

axes[1].hist(aedxai_df["iterations"], bins=np.arange(aedxai_df["iterations"].max() + 2) - 0.5, color="#C4AD66", edgecolor="black")
axes[1].set_xlabel("num_iterations")
axes[1].set_ylabel("image count")
axes[1].set_title("AED-XAI Iterations")

scatter = axes[2].scatter(aedxai_df["mean_pg"], aedxai_df["mean_oa"], c=aedxai_df["composite_score"], cmap="viridis", label="AED-XAI", alpha=0.8)
axes[2].scatter(baseline_df["mean_pg"], baseline_df["mean_oa"], c=baseline_df["composite_score"], cmap="viridis", marker="x", label="GradCAM", alpha=0.8)
axes[2].set_xlabel("PG")
axes[2].set_ylabel("OA")
axes[2].set_title("PG vs OA")
axes[2].legend()
fig.colorbar(scatter, ax=axes[2], label="Composite")

plt.tight_layout()
plt.show()

## Save Everything

In [ ]:
results_dir = PROJECT_ROOT / "results"
figures_dir = results_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

aedxai_df.to_csv(results_dir / "notebook_aedxai.csv", index=False)
baseline_df.to_csv(results_dir / "notebook_baseline.csv", index=False)
fig.savefig(figures_dir / "comparison.png", dpi=150, bbox_inches="tight")

pipeline.shutdown()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Done.")